# Consultas en vivo con trazas en Phoenix

Cada consulta que se haga aquí aparece en **Phoenix** (http://localhost:6006 → proyecto `clausefinder-crew`)
como un árbol de trazas: agente → llamadas al LLM → llamadas a tools, con prompts, tokens, costo y latencia.

**Antes de empezar**
1. Kernel: **Python (ClauseFinder Crew)** (menú *Kernel → Change kernel*). Es el Python de `.venv-crew`,
   donde están CrewAI, LangChain y la instrumentación.
2. Phoenix corriendo: `python -m src.observabilidad` en una terminal (o el panel de vista previa).
3. `DEEPSEEK_API_KEY` en `.env`. Cada pregunta del chat cuesta del orden de US$ 0,002; una revisión
   completa del crew, unos US$ 0,10.

In [2]:
import pathlib, sys

RAIZ = pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
if ".venv-crew" not in sys.executable:
    raise RuntimeError("Este notebook necesita el kernel 'Python (ClauseFinder Crew)' (Kernel → Change kernel).")

# Activar ANTES de crear agentes: instrumenta CrewAI, LangChain y el SDK openai.
from src.observabilidad import URL_UI, activar, enviar_trazas
activar()
print(f"Trazas → {URL_UI}  (proyecto clausefinder-crew)")

Trazas → http://localhost:6006  (proyecto clausefinder-crew)


## 1. Chat de consulta (LangChain) — una conversación = una sesión en Phoenix

Todas las preguntas de `chat` comparten `session.id`: en Phoenix, pestaña **Sessions**, se ve la
conversación completa turno a turno. La memoria de ventana (k=4) permite preguntas de seguimiento.

In [3]:
from src.chat.consulta import ConsultaContrato

chat = ConsultaContrato("MSA_MeritLife_2020", k=4)
print("Sesión en Phoenix:", chat.id_sesion)

Sesión en Phoenix: consulta-MSA_MeritLife_2020-90602d


In [4]:
respuesta, pasos = chat.preguntar("¿El contrato limita la responsabilidad del contratista? ¿En qué sección?")
print(respuesta)
print(f"\n[{len(pasos)} llamadas a tools]")
enviar_trazas()   # sin esto aparecen igual, a los ~5 s

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2454.00it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2307.36it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can 

Sí. La cláusula está en la **Sección 7, "LIMITATIONS OF LIABILITY; INDEMNIFICATION OF CONTRACTOR"** (pág. 7):

- **Tope para el contratista:** "EXCEPT WITH RESPECT TO CONTRACTOR'S OBLIGATIONS PURSUANT TO PARAGRAPH 9 HEREOF, CONTRACTOR'S MAXIMUM LIABILITY TO COMPANY ARISING FOR ANY REASON RELATING TO CONTRACTOR'S PERFORMANCE OF SERVICES UNDER A STATEMENT OF WORK SHALL BE LIMITED TO THE AMOUNT OF FEES PAID TO CONTRACTOR FOR THE PERFORMANCE OF SUCH SERVICES." (Sec. 7(a))
- **Tope recíproco para la empresa:** "COMPANYS' MAXIMUM LIABILITY TO CONTRACTOR FOR ANY REASON ARISING OUT OF THIS AGREEMENT SHALL BE LIMITED TO THE AMOUNT OF FEES PAID TO CONTRACTOR." (Sec. 7(a))
- **Exclusión de daños:** "NEITHER PARTY SHALL HAVE ANY LIABILITY TO THE OTHER PARTY FOR ANY LOST PROFITS OR SPECIAL, INCIDENTAL, INDIRECT OR CONSEQUENTIAL DAMAGES, EVEN IF SUCH PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES." (Sec. 7(b))
- **Honorarios de abogados:** "IN ANY SUIT ARISING FROM THIS AGREEMENT EACH PAR

In [5]:
respuesta, pasos = chat.preguntar("¿Esa cláusula tiene alguna excepción?")
print(respuesta)
enviar_trazas()

Sí, hay una excepción expresa al tope del contratista, y es la única que el texto formula como tal:

- **Excepción al límite del contratista:** "EXCEPT WITH RESPECT TO CONTRACTOR'S OBLIGATIONS PURSUANT TO PARAGRAPH 9 HEREOF, CONTRACTOR'S MAXIMUM LIABILITY TO COMPANY ... SHALL BE LIMITED TO THE AMOUNT OF FEES PAID TO CONTRACTOR FOR THE PERFORMANCE OF SUCH SERVICES." (Sec. 7(a))

Es decir, el límite no aplica a las obligaciones del contratista bajo el Párrafo 9. Ese párrafo es *Independent Contractor* (pág. 8) y su obligación económica principal es: "Subject to Section 3(c), Contractor shall be responsible for all taxes and other expenses arising from the employment or independent contractor relationship between Contractor and its personnel and the rendition of Services hereunder by such personnel to Company." (Sec. 9)

Otros puntos a tener en cuenta:
- El tope de la empresa no lleva excepción: "COMPANYS' MAXIMUM LIABILITY TO CONTRACTOR FOR ANY REASON ARISING OUT OF THIS AGREEMENT SHALL 

In [6]:
# Haz tus propias preguntas: cambia el texto y vuelve a ejecutar esta celda.
respuesta, pasos = chat.preguntar("¿Se puede ceder el contrato a un tercero?")
print(respuesta)
enviar_trazas()

No libremente: requiere consentimiento escrito previo de la otra parte.

- **Sección 10, "NONASSIGNABILITY"** (pág. 8): "Neither party shall assign, transfer, or subcontract this Agreement or any of its obligations hereunder without the other party's express, prior written consent, which will not be unreasonably withheld."

Es decir, la restricción es mutua, cubre también la subcontratación de obligaciones y el consentimiento no puede negarse de forma irrazonable. No se prevé una excepción para cesiones a afiliadas ni por cambio de control.

No soy asesor legal: esto solo describe lo que dice el contrato.


## 2. Agente único desde cero (function calling)

Revisa un contrato contra el checklist de 10 categorías. En Phoenix: un span `agente function calling`
con sus llamadas al LLM y cada tool como hijo (~1-2 min, ~US$ 0,007).

In [7]:
from src.agents_scratch.function_calling import ejecutar_fc

c = ejecutar_fc("Mantenimiento_XImage_Sagem_1999")
for d in c.respuesta["determinaciones"]:
    print(f"  {d['categoria']:<28} {'Sí §' + str(d['seccion']) if d['presente'] else 'No'}")
enviar_trazas()

  uncapped_liability           No
  cap_on_liability             Sí §15
  liquidated_damages           No
  termination_for_convenience  No
  renewal_term                 Sí §2
  anti_assignment              Sí §18
  governing_law                No
  exclusivity                  No
  non_compete                  No
  ip_ownership_assignment      Sí §3


## 3. El crew completo (Auditor ×3 → Defensor → Verificador → Redactor)

En Phoenix: span raíz `revisión <contrato> · C3` → Flow → crews → cada agente → LLM y tools MCP.
Levanta el servidor MCP si no está corriendo (~50 s) y tarda unos minutos (~US$ 0,10).
Con el MSA suele verse el ciclo rechazo → reintento → aprobación.

In [8]:
from src.crew.flow import revisar_contrato

estado, carpeta = revisar_contrato("MSA_MeritLife_2020", "C3")
for categoria, f in estado.finales.items():
    print(f"  {categoria:<28} {f.estado}")
print("\nEventos del Flow:")
for ev in estado.eventos:
    print("  ", ev[:140])
enviar_trazas()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: FlujoRevision                                                                                            │
│  ID: 725c29d7-1dcb-4588-b3c9-d7aa03fd76f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.26                                                                                       │
│  Latest version:  1.15.29                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: FlujoRevision                                                                                            │
│  ID: 725c29d7-1dcb-4588-b3c9-d7aa03fd76f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


>>> [flow] revisión de MSA_MeritLife_2020 con Config(nombre='C3', defensor=True, reintentos=1, jerarquico=False, concurrente=True, memoria=False)



╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: revisar                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b9c63ea4-c748-4646-aaa3-d53ac85f9616                                                                       │
│  Final Output: Informe guardado correctamente.                                                                  │
│                                                                                                                 │
│  **Ruta:** `outputs/informes/MSA_MeritLife_2020_20261023_201400.md` (6 citas verificadas)                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Informe de riesgos — Contrato MSA_MeritLife_2020                                                             │
│                                                                                                                 │
│  ## Resumen ejecutivo                                                                                           │
│                                                                                                                 │
│  El contrato presenta un riesgo de severidad Alta derivado de la excepción al tope de responsabilidad del       │
│  Contratista respecto de sus obligaciones del Párrafo 9, que abre una responsabilidad sin límite cuantitativo   │
│  frente a impuestos, gastos y actos de su propio personal. Un segundo hallazgo de severidad Media afecta a la   │
│  titularidad de la propiedad intelectual sobre los Works creados para la Compañía. Los cuatro hallazgos         │
│  restantes son de severidad Baja y se refieren al tope de responsabilidad, la terminación por conveniencia, la  │
│  prohibición de cesión y la ley aplicable.                                                                      │
│                                                                                                                 │
│  ## Tabla de riesgos                                                                                            │
│                                                                                                                 │
│  | Categoría | Severidad | Sección |                                                                            │
│  |---|---|---|                                                                                                  │
│  | Responsabilidad ilimitada | Alta | 7 |                                                                       │
│  | Cesión de propiedad intelectual | Media | 4 |                                                                │
│  | Tope de responsabilidad | Baja | 7 |                                                                         │
│  | Terminación por conveniencia | Baja | 5 |                                                                    │
│  | Prohibición de cesión | Baja | 10 |                                                                          │
│  | Ley aplicable | Baja | 11 |                                                                                  │
│                                                                                                                 │
│  ## Hallazgos                                                                                                   │
│                                                       


>>> [flow] Redactor: Informe guardado correctamente.

**Ruta:** `outputs/informes/MSA_MeritLife_2020_20261023_201400.md` (6 citas verificadas)

---

# Informe de riesgos — Contrato MSA_MeritLife_2020

## Resumen ejecutivo



╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: redactar                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: FlujoRevision                                                                                            │
│  ID: 725c29d7-1dcb-4588-b3c9-d7aa03fd76f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  uncapped_liability           aprobado
  cap_on_liability             aprobado
  liquidated_damages           ausente
  termination_for_convenience  aprobado
  renewal_term                 ausente
  anti_assignment              aprobado
  governing_law                aprobado
  exclusivity                  ausente
  non_compete                  ausente
  ip_ownership_assignment      aprobado

Eventos del Flow:
   20:10:41 revisión de MSA_MeritLife_2020 con Config(nombre='C3', defensor=True, reintentos=1, jerarquico=False, concurrente=True, memoria=Fal
   20:12:57 ronda 0: {'aprobado': 5, 'rechazado': 1, 'descartado': 0, 'ausente': 4}
   20:12:57 reintento 1: ['uncapped_liability']
   20:13:49 ronda 1: {'aprobado': 1, 'rechazado': 0, 'descartado': 0, 'ausente': 0}
   20:14:08 Redactor: Informe guardado correctamente.

**Ruta:** `outputs/informes/MSA_MeritLife_2020_20261023_201400.md` (6 citas verificadas)
